### Step 4: Model Preprocessing

Preprocess the data for modeling. Split the data into train, val, test datasets

In [5]:
import pandas as pd

from sklearn.preprocessing import StandardScaler, OrdinalEncoder

def feature_engineered_filepath(res: str) -> str:
    return f"../data/003_feature_engineered/{res}_power_load.parquet"

def preprocess_filepath(res: str, split: str) -> str:
    return f"../data/004_preprocessed/{res}/{split}.parquet"

RESAMPLE_RESOLUTIONS = ["30s", "1min", "2min", "5min", "10min", "15min"]
VAL_DAYS  = 3
TEST_DAYS = 3
CATEGORICAL_COLS = ["workday", "time_of_day"]
DROP_COLS        = ["timestamp", "load"]

In [6]:
def preprocess(df: pd.DataFrame, scale: bool = True):
    
    # 1. Drop NaNs
    df = df.dropna().copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # 2. Encode categoricals
    enc = OrdinalEncoder()
    df[CATEGORICAL_COLS] = enc.fit_transform(df[CATEGORICAL_COLS])

    # 3. Dynamic split based on last N days
    max_date = df["timestamp"].max()
    test_start = max_date - pd.Timedelta(days=TEST_DAYS)
    val_start  = test_start - pd.Timedelta(days=VAL_DAYS)

    df_train = df[df["timestamp"] <= val_start]
    df_val   = df[(df["timestamp"] > val_start) & (df["timestamp"] <= test_start)]
    df_test  = df[df["timestamp"] > test_start]

    print(f"Train: {df_train.shape} | {df['timestamp'].min().date()} → {val_start.date()}")
    print(f"Val:   {df_val.shape}   | {val_start.date()} → {test_start.date()}")
    print(f"Test:  {df_test.shape}  | {test_start.date()} → {max_date.date()}")

    # 4. X / y split
    X_train, y_train = df_train.drop(columns=DROP_COLS), df_train["load"]
    X_val,   y_val   = df_val.drop(columns=DROP_COLS),   df_val["load"]
    X_test,  y_test  = df_test.drop(columns=DROP_COLS),  df_test["load"]

    # 5. Scale (fit on train only)
    if scale:
        numeric_cols = [c for c in X_train.columns if c not in CATEGORICAL_COLS]
        scaler = StandardScaler()
        X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
        X_val[numeric_cols]   = scaler.transform(X_val[numeric_cols])
        X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

    y_train = y_train.to_frame()
    y_val   = y_val.to_frame()
    y_test  = y_test.to_frame()

    return X_train, y_train, X_val, y_val, X_test, y_test


# ── RUN ───────────────────────────────────────────────────────────────────────
for res in RESAMPLE_RESOLUTIONS:
    print(f"\nPreprocessing {res} dataset:")
    df = pd.read_parquet(feature_engineered_filepath(res))
    X_train, y_train, X_val, y_val, X_test, y_test = preprocess(df, scale=True)
    X_train.to_parquet(preprocess_filepath(res, "X_train"), index=False)
    y_train.to_parquet(preprocess_filepath(res, "y_train"), index=False)
    X_val.to_parquet(preprocess_filepath(res, "X_val"), index=False)
    y_val.to_parquet(preprocess_filepath(res, "y_val"), index=False)
    X_test.to_parquet(preprocess_filepath(res, "X_test"), index=False)
    y_test.to_parquet(preprocess_filepath(res, "y_test"), index=False)


Preprocessing 30s dataset:
Train: (71222, 44) | 2025-11-28 → 2025-12-22
Val:   (8406, 44)   | 2025-12-22 → 2025-12-25
Test:  (8640, 44)  | 2025-12-25 → 2025-12-28

Preprocessing 1min dataset:
Train: (35444, 44) | 2025-11-28 → 2025-12-22
Val:   (4176, 44)   | 2025-12-22 → 2025-12-25
Test:  (4320, 44)  | 2025-12-25 → 2025-12-28

Preprocessing 2min dataset:
Train: (17756, 44) | 2025-11-28 → 2025-12-22
Val:   (2029, 44)   | 2025-12-22 → 2025-12-25
Test:  (2160, 44)  | 2025-12-25 → 2025-12-28

Preprocessing 5min dataset:
Train: (7140, 44) | 2025-11-28 → 2025-12-22
Val:   (802, 44)   | 2025-12-22 → 2025-12-25
Test:  (864, 44)  | 2025-12-25 → 2025-12-28

Preprocessing 10min dataset:
Train: (3540, 44) | 2025-11-28 → 2025-12-22
Val:   (371, 44)   | 2025-12-22 → 2025-12-25
Test:  (432, 44)  | 2025-12-25 → 2025-12-28

Preprocessing 15min dataset:
Train: (2340, 44) | 2025-11-28 → 2025-12-22
Val:   (288, 44)   | 2025-12-22 → 2025-12-25
Test:  (288, 44)  | 2025-12-25 → 2025-12-28
